# AutoGen Tools & Code Execution

## Course Notebook for Students

**Goal:** Learn how AutoGen agents use tools, call Python functions, execute code, and solve real-world tasks safely.

This notebook is designed for a hands-on lecture. It mixes theory, runnable examples, mini-projects, and exercises.

> Note: AutoGen is currently maintained by the community, and Microsoft recommends Microsoft Agent Framework for new long-term enterprise projects. This course still focuses on AutoGen because the topic is AutoGen Tools & Code Execution.

### What students will build

1. A simple tool and direct tool execution.
2. An AutoGen assistant that uses business tools.
3. A retail inventory and offer assistant.
4. A code-execution tool that runs Python.
5. A sales-data analysis agent that creates files and insights.
6. A multi-agent workflow for planning, analysis, and reporting.
7. A safe approval pattern before executing generated code.

## 0. Prerequisites

Students should know:

- Basic Python functions
- Dictionaries and lists
- `async` / `await` basics
- Environment variables
- Basic LLM concepts: prompt, model, token, tool call

### Recommended setup

- Python 3.10+
- Jupyter Notebook or VS Code notebook
- OpenAI API key stored as `OPENAI_API_KEY`

### Important safety rule

Never run untrusted generated code on your main computer without review. Prefer Docker or a disposable environment for serious projects.

In [ ]:
# Run this cell once in a fresh environment.
# If you already installed these packages, you can skip it.

#%pip install -U "autogen-agentchat" "autogen-ext[openai]" pandas matplotlib typing_extensions pydantic

In [5]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("Python:", sys.version)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

WORK_DIR = Path("autogen_course_workspace")
WORK_DIR.mkdir(exist_ok=True)
print("Workspace:", WORK_DIR.resolve())

Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:36:12) [MSC v.1944 64 bit (AMD64)]
OPENAI_API_KEY set: True
Workspace: C:\Users\LotusBlue\Coding\GenAI\Course\autogen_course_workspace


## 1. Mental Model: What Is a Tool?

In AutoGen, a **tool** is code an agent can call to perform an action.

Examples:

- Check product stock
- Calculate discount
- Query a database
- Call a REST API
- Execute Python code
- Create a chart
- Read a file

An LLM decides *when* to call the tool. The tool itself performs the real action.

### Tool call flow
** Most important

User task -> Agent reasons -> Agent emits tool call -> AutoGen runs tool -> Tool result goes back to agent -> Agent responds

In [7]:
from autogen_core import CancellationToken
from autogen_core.tools import FunctionTool
from typing_extensions import Annotated


async def calculate_offer_price(
    mrp: Annotated[float, "Original marked retail price in INR"],
    discount_percent: Annotated[float, "Discount percentage, for example 15 for 15%"],
) -> dict:
    """Calculate final selling price after discount for a retail product."""
    if mrp < 0:
        raise ValueError("MRP cannot be negative")
    if not 0 <= discount_percent <= 90:
        raise ValueError("Discount percent must be between 0 and 90")

    discount_amount = round(mrp * discount_percent / 100, 2)
    final_price = round(mrp - discount_amount, 2)
    return {
        "mrp": mrp,
        "discount_percent": discount_percent,
        "discount_amount": discount_amount,
        "final_price": final_price,
    }


offer_tool = FunctionTool(
    calculate_offer_price,
    description="Calculate final selling price after applying discount.",
)

print(offer_tool.schema)

{'name': 'calculate_offer_price', 'description': 'Calculate final selling price after applying discount.', 'parameters': {'type': 'object', 'properties': {'mrp': {'description': 'Original marked retail price in INR', 'title': 'Mrp', 'type': 'number'}, 'discount_percent': {'description': 'Discount percentage, for example 15 for 15%', 'title': 'Discount Percent', 'type': 'number'}}, 'required': ['mrp', 'discount_percent'], 'additionalProperties': False}, 'strict': False}


In [9]:
# Directly run the tool without an LLM.
# This helps students understand that tools are normal executable code.

result = await offer_tool.run_json(
    {"mrp": 2499, "discount_percent": 20},
    CancellationToken(),
)

print(offer_tool.return_value_as_string(result))

{'mrp': 2499.0, 'discount_percent': 20.0, 'discount_amount': 499.8, 'final_price': 1999.2}


## 2. AutoGen Assistant with Tools

Now we connect tools to an `AssistantAgent`.

This example uses a retail store use case:

- Check inventory
- Calculate discount
- Explain store policy

The agent chooses the right tool based on the customer question.

In [11]:
import os
from autogen_ext.models.openai import OpenAIChatCompletionClient


def require_openai_key() -> None:
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError(
            "OPENAI_API_KEY is not set. Set it before running LLM cells."
        )


def create_model_client(model: str = "gpt-4o-mini") -> OpenAIChatCompletionClient:
    require_openai_key()
    return OpenAIChatCompletionClient(model=model)

In [13]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console


INVENTORY = {
    "cotton kurti": {"stock": 18, "price": 1299, "sizes": ["S", "M", "L", "XL"]},
    "linen co-ord set": {"stock": 7, "price": 2499, "sizes": ["M", "L"]},
    "rayon salwar set": {"stock": 12, "price": 1899, "sizes": ["S", "M", "XL"]},
}


def check_inventory(product_name: str) -> dict:
    """Check store inventory, available sizes, and price for a product name."""
    product_name = product_name.lower().strip()
    return INVENTORY.get(
        product_name,
        {"stock": 0, "message": f"No exact match found for {product_name}"},
    )


def calculate_bill(quantity: int, unit_price: float, discount_percent: float = 0) -> dict:
    """Calculate retail bill after discount for a given quantity and unit price."""
    subtotal = quantity * unit_price
    discount = subtotal * discount_percent / 100
    return {
        "quantity": quantity,
        "unit_price": unit_price,
        "subtotal": round(subtotal, 2),
        "discount": round(discount, 2),
        "payable": round(subtotal - discount, 2),
    }


def store_policy(topic: str) -> str:
    """Return store policy for exchange, timing, alteration, or offers."""
    policies = {
        "exchange": "Exchange is allowed within 7 days with bill and original tag.",
        "timing": "Store timing is 10:30 AM to 9:00 PM.",
        "alteration": "Basic alteration is available for selected products.",
        "offers": "Current offer: Buy 3, Get 1 on selected kurti and salwar sets.",
    }
    return policies.get(topic.lower().strip(), "Policy not found for this topic.")

In [15]:
# LLM cell: requires OPENAI_API_KEY.

model_client = create_model_client()

retail_agent = AssistantAgent(
    name="retail_assistant",
    model_client=model_client,
    tools=[check_inventory, calculate_bill, store_policy],
    system_message=(
        "You are a helpful retail store assistant. "
        "Use tools for inventory, billing, and policy questions. "
        "Give concise answers in friendly language."
    ),
    reflect_on_tool_use=True,
)

await Console(
    retail_agent.run_stream(
        task=(
            "A customer wants 2 linen co-ord sets. "
            "Check availability, calculate bill with 10% discount, "
            "and mention exchange policy."
        )
    )
)

---------- TextMessage (user) ----------
A customer wants 2 linen co-ord sets. Check availability, calculate bill with 10% discount, and mention exchange policy.
---------- ToolCallRequestEvent (retail_assistant) ----------
[FunctionCall(id='call_QlRb3CnqthGO2pwdCujnnsdS', arguments='{"product_name": "linen co-ord set"}', name='check_inventory'), FunctionCall(id='call_sC9ozB5MXtuEXLG0ZQUhLiOR', arguments='{"topic": "exchange"}', name='store_policy')]
---------- ToolCallExecutionEvent (retail_assistant) ----------
[FunctionExecutionResult(content="{'stock': 7, 'price': 2499, 'sizes': ['M', 'L']}", name='check_inventory', call_id='call_QlRb3CnqthGO2pwdCujnnsdS', is_error=False), FunctionExecutionResult(content='Exchange is allowed within 7 days with bill and original tag.', name='store_policy', call_id='call_sC9ozB5MXtuEXLG0ZQUhLiOR', is_error=False)]
---------- TextMessage (retail_assistant) ----------
We have 7 linen co-ord sets available, priced at ₹2499 each. For 2 sets, the total co

TaskResult(messages=[TextMessage(id='34e30412-5d4b-4f1d-8912-8b6cd383c218', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 16, 45, 602918, tzinfo=datetime.timezone.utc), content='A customer wants 2 linen co-ord sets. Check availability, calculate bill with 10% discount, and mention exchange policy.', type='TextMessage'), ToolCallRequestEvent(id='3ac55880-9bf4-4288-80da-a51ec0096953', source='retail_assistant', models_usage=RequestUsage(prompt_tokens=210, completion_tokens=49), metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 16, 48, 852006, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_QlRb3CnqthGO2pwdCujnnsdS', arguments='{"product_name": "linen co-ord set"}', name='check_inventory'), FunctionCall(id='call_sC9ozB5MXtuEXLG0ZQUhLiOR', arguments='{"topic": "exchange"}', name='store_policy')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='3bdda111-d6e9-42a4-b927-693033730506', source='retail_assistant', mod

## 3. Tool Design Best Practices

Good tools are:

- Small and specific
- Strongly typed
- Clearly described
- Validated
- Deterministic where possible
- Safe to execute repeatedly

Avoid tools like:

- `do_everything(input: str)`
- `run_any_shell_command(command: str)`
- Tools that delete data without approval
- Tools that silently call paid APIs

In [17]:
def reserve_stock(product_name: str, quantity: int) -> dict:
    """Reserve stock for a product if enough quantity is available."""
    if quantity <= 0:
        raise ValueError("Quantity must be positive")

    product_key = product_name.lower().strip()
    item = INVENTORY.get(product_key)
    if not item:
        return {"reserved": False, "reason": "Product not found"}

    if item["stock"] < quantity:
        return {
            "reserved": False,
            "reason": "Insufficient stock",
            "available_stock": item["stock"],
        }

    item["stock"] -= quantity
    return {
        "reserved": True,
        "product": product_key,
        "quantity": quantity,
        "remaining_stock": item["stock"],
    }


print(reserve_stock("cotton kurti", 2))
print(reserve_stock("cotton kurti", 100))

{'reserved': True, 'product': 'cotton kurti', 'quantity': 2, 'remaining_stock': 16}
{'reserved': False, 'reason': 'Insufficient stock', 'available_stock': 16}


## 4. Code Execution in AutoGen

Code execution lets an agent write and run Python code.

This is useful for:

- Data cleaning
- CSV analysis
- Chart creation
- Mathematical calculations
- File generation
- Testing generated code

### Local vs Docker execution

**Local execution** runs on your machine. It is easier for class demos but less safe.

**Docker execution** runs inside a container. It is safer for untrusted generated code and recommended for serious projects.

In [19]:
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor
from autogen_ext.tools.code_execution import PythonCodeExecutionTool


code_executor = LocalCommandLineCodeExecutor(work_dir=str(WORK_DIR / "code_execution"))
python_code_tool = PythonCodeExecutionTool(code_executor)

sample_code = '''
sales = [1200, 1500, 900, 2100, 1800]
print("Total sales:", sum(sales))
print("Average sales:", round(sum(sales) / len(sales), 2))
'''

execution_result = await python_code_tool.run_json(
    {"code": sample_code},
    CancellationToken(),
)

print(python_code_tool.return_value_as_string(execution_result))

Total sales: 7500
Average sales: 1500.0



C:\Users\LotusBlue\AppData\Local\Temp\ipykernel_3184\3892799886.py:5: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  code_executor = LocalCommandLineCodeExecutor(work_dir=str(WORK_DIR / "code_execution"))


In [21]:
# Real-world use case: generate and analyze a small retail sales dataset.

sales_analysis_code = '''
import pandas as pd
from pathlib import Path

data = [
    {"date": "2026-06-01", "category": "Kurti", "units": 12, "revenue": 15588},
    {"date": "2026-06-01", "category": "Co-ord Set", "units": 4, "revenue": 9996},
    {"date": "2026-06-02", "category": "Kurti", "units": 9, "revenue": 11691},
    {"date": "2026-06-02", "category": "Salwar Set", "units": 6, "revenue": 11394},
    {"date": "2026-06-03", "category": "Co-ord Set", "units": 5, "revenue": 12495},
]

df = pd.DataFrame(data)
summary = df.groupby("category", as_index=False).agg(
    total_units=("units", "sum"),
    total_revenue=("revenue", "sum"),
)
summary["avg_price"] = (summary["total_revenue"] / summary["total_units"]).round(2)

output_path = Path("category_sales_summary.csv")
summary.to_csv(output_path, index=False)

print(summary)
print(f"Saved file: {output_path.resolve()}")
'''

result = await python_code_tool.run_json(
    {"code": sales_analysis_code},
    CancellationToken(),
)

print(python_code_tool.return_value_as_string(result))

     category  total_units  total_revenue  avg_price
0  Co-ord Set            9          22491     2499.0
1       Kurti           21          27279     1299.0
2  Salwar Set            6          11394     1899.0
Saved file: C:\Users\LotusBlue\Coding\GenAI\Course\autogen_course_workspace\code_execution\category_sales_summary.csv



## 5. Assistant Agent + Python Code Execution Tool

In this pattern, the agent receives a business task, writes Python code, executes it through the tool, observes the result, and explains the answer.

This is the core idea behind many data-analysis agents.

In [24]:
# LLM cell: requires OPENAI_API_KEY.

analysis_agent = AssistantAgent(
    name="sales_analysis_agent",
    model_client=create_model_client(),
    tools=[python_code_tool],
    system_message=(
        "You are a careful retail data analyst. "
        "When calculation is needed, write and run Python code using the code execution tool. "
        "Use only small safe Python snippets. "
        "Explain the result clearly for a store owner."
    ),
    reflect_on_tool_use=True,
)

task = '''
Create a small pandas DataFrame for 7 days of store sales with columns:
date, walkins, buyers, revenue.
Then calculate:
1. conversion rate per day
2. total revenue
3. best day by revenue
4. one practical recommendation
'''

await Console(analysis_agent.run_stream(task=task))

---------- TextMessage (user) ----------

Create a small pandas DataFrame for 7 days of store sales with columns:
date, walkins, buyers, revenue.
Then calculate:
1. conversion rate per day
2. total revenue
3. best day by revenue
4. one practical recommendation

---------- ToolCallRequestEvent (sales_analysis_agent) ----------
[FunctionCall(id='call_d1UkflKnkbihVuqjVfMvYF5u', arguments='{"code":"import pandas as pd\\n\\n# Creating a small DataFrame for 7 days of store sales\\ndata = {\\n    \'date\': [\'2023-10-01\', \'2023-10-02\', \'2023-10-03\', \'2023-10-04\', \'2023-10-05\', \'2023-10-06\', \'2023-10-07\'],\\n    \'walkins\': [100, 150, 120, 130, 90, 110, 140],\\n    \'buyers\': [30, 45, 35, 40, 25, 55, 50],\\n    \'revenue\': [300, 450, 350, 400, 250, 550, 500]\\n}\\n\\nsales_df = pd.DataFrame(data)\\n\\nsales_df[\'conversion_rate\'] = sales_df[\'buyers\'] / sales_df[\'walkins\'] * 100\\n\\n# Calculating total revenue and best day by revenue\\ntotal_revenue = sales_df[\'revenue\']

TaskResult(messages=[TextMessage(id='6aa59d80-8a40-4908-8292-f8f099c87c8d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 17, 49, 293031, tzinfo=datetime.timezone.utc), content='\nCreate a small pandas DataFrame for 7 days of store sales with columns:\ndate, walkins, buyers, revenue.\nThen calculate:\n1. conversion rate per day\n2. total revenue\n3. best day by revenue\n4. one practical recommendation\n', type='TextMessage'), ToolCallRequestEvent(id='1246ed8d-75c7-4f87-8f38-47615d504dd8', source='sales_analysis_agent', models_usage=RequestUsage(prompt_tokens=153, completion_tokens=270), metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 17, 53, 94029, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_d1UkflKnkbihVuqjVfMvYF5u', arguments='{"code":"import pandas as pd\\n\\n# Creating a small DataFrame for 7 days of store sales\\ndata = {\\n    \'date\': [\'2023-10-01\', \'2023-10-02\', \'2023-10-03\', \'2023-10-04\', \'20

## 6. Optional: Safer Docker Code Execution

Use this only if Docker is installed and running.

Why Docker?

- Keeps generated code away from your main environment
- Makes dependencies more reproducible
- Easier cleanup after execution


In [26]:
# Optional Docker example.
# Run only if Docker is installed and running.

RUN_DOCKER_EXAMPLE = False

if RUN_DOCKER_EXAMPLE:
    from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor

    async with DockerCommandLineCodeExecutor(work_dir=str(WORK_DIR / "docker_execution")) as docker_executor:
        docker_tool = PythonCodeExecutionTool(docker_executor)
        docker_result = await docker_tool.run_json(
            {"code": "print('Hello from Docker-based AutoGen code execution')"},
            CancellationToken(),
        )
        print(docker_tool.return_value_as_string(docker_result))
else:
    print("Docker example skipped. Set RUN_DOCKER_EXAMPLE = True to run it.")

Docker example skipped. Set RUN_DOCKER_EXAMPLE = True to run it.


## 7. Human Approval Pattern Before Code Execution

In production, do not let an LLM run arbitrary code freely.

A simple classroom pattern:

1. Ask the agent to propose code.
2. Show code to a human.
3. Human approves.
4. Execute code.

Below is a simple helper that blocks risky keywords. This is not complete security, but it teaches the right idea.

In [3]:
RISKY_KEYWORDS = [
    "import os",
    "import subprocess",
    "shutil.rmtree",
    "rm -rf",
    "open('/",
    "requests.post",
    "socket",
]


def simple_code_safety_check(code_text: str) -> tuple[bool, list[str]]:
    """Return whether code passed a simple safety check and which patterns were flagged."""
    lowered = code_text.lower()
    flagged = [kw for kw in RISKY_KEYWORDS if kw.lower() in lowered]
    return len(flagged) == 0, flagged


candidate_code = '''
import pandas as pd
df = pd.DataFrame({"sales": [1000, 1500, 2000]})
print(df["sales"].sum())
'''

passed, flagged = simple_code_safety_check(candidate_code)
print("Passed:", passed)
print("Flagged:", flagged)

if passed:
    result = await python_code_tool.run_json({"code": candidate_code}, CancellationToken())
    print(python_code_tool.return_value_as_string(result))

Passed: True
Flagged: []


NameError: name 'python_code_tool' is not defined

## 8. Multi-Agent Workflow: Planner + Analyst + Reporter

This example uses a `SelectorGroupChat`.

Real-world scenario:

> A retail owner wants to understand why sales are low and what action to take.

Agents:

- **PlanningAgent** breaks the task into steps.
- **DataAnalystAgent** uses Python code execution for calculation.
- **ReportAgent** converts results into a business-friendly summary.

In [31]:
# LLM cell: requires OPENAI_API_KEY.

from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import SelectorGroupChat


model_client = create_model_client()

planning_agent = AssistantAgent(
    "PlanningAgent",
    description="Breaks retail analysis tasks into clear steps and decides when the task is complete.",
    model_client=model_client,
    system_message=(
        "You are a planning agent. Break the user request into steps. "
        "Assign data tasks to DataAnalystAgent and final summary tasks to ReportAgent. "
        "When everything is complete, say TERMINATE."
    ),
)

data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="Runs Python analysis for retail sales, conversion, and revenue questions.",
    model_client=model_client,
    tools=[python_code_tool],
    system_message=(
        "You are a data analyst. Use the Python code execution tool for calculations. "
        "Keep datasets small and safe for classroom demonstration."
    ),
    reflect_on_tool_use=True,
)

report_agent = AssistantAgent(
    "ReportAgent",
    description="Writes simple business recommendations from analysis results.",
    model_client=model_client,
    system_message=(
        "You are a business report writer. Convert analysis into clear store-owner recommendations. "
        "Be concise and practical."
    ),
)

termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=12)

selector_prompt = '''
Select the next agent to perform the task.

{roles}

Current conversation context:
{history}

Choose one agent from {participants}.
Planner should coordinate. Analyst should calculate. Reporter should summarize.
'''

team = SelectorGroupChat(
    [planning_agent, data_analyst_agent, report_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
)

await Console(
    team.run_stream(
        task=(
            "Analyze a small sample of 7 days of boutique sales. "
            "Include walk-ins, buyers, and revenue. Find conversion rate, "
            "best day, weakest day, and recommend 3 actions."
        )
    )
)

---------- TextMessage (user) ----------
Analyze a small sample of 7 days of boutique sales. Include walk-ins, buyers, and revenue. Find conversion rate, best day, weakest day, and recommend 3 actions.
---------- TextMessage (PlanningAgent) ----------
To fulfill your request, the tasks can be broken down into the following steps:

### Step 1: Data Collection
- **Task for DataAnalystAgent:** Gather data for the past 7 days of boutique sales, including number of walk-ins, number of buyers, and total revenue.

### Step 2: Data Analysis
- **Task for DataAnalystAgent:** Calculate the conversion rate (conversion rate = number of buyers / number of walk-ins).
- **Task for DataAnalystAgent:** Identify the best day (highest revenue) and weakest day (lowest revenue) from the 7-day sample.

### Step 3: Generate Recommendations
- **Task for DataAnalystAgent:** Analyze trends and customer behavior to recommend 3 actions aimed at improving sales or enhancing customer experience.

### Step 4: Report 

TaskResult(messages=[TextMessage(id='b52caca9-f5ad-4117-9f58-35dea6132390', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 18, 33, 164322, tzinfo=datetime.timezone.utc), content='Analyze a small sample of 7 days of boutique sales. Include walk-ins, buyers, and revenue. Find conversion rate, best day, weakest day, and recommend 3 actions.', type='TextMessage'), TextMessage(id='9b6daddb-3d6b-4590-a7a3-71a844f832a8', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=87, completion_tokens=222), metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 18, 41, 687893, tzinfo=datetime.timezone.utc), content='To fulfill your request, the tasks can be broken down into the following steps:\n\n### Step 1: Data Collection\n- **Task for DataAnalystAgent:** Gather data for the past 7 days of boutique sales, including number of walk-ins, number of buyers, and total revenue.\n\n### Step 2: Data Analysis\n- **Task for DataAnalystAgent:** Cal

## 9. Mini Project 1: Customer Support Tool Agent

Build an AutoGen assistant that can:

- Classify ticket priority
- Suggest next action
- Estimate SLA

Students can extend this into a real helpdesk assistant.

In [32]:
def classify_ticket(message: str) -> dict:
    """Classify a customer support ticket into priority and category."""
    text = message.lower()
    if "payment" in text or "refund" in text:
        return {"priority": "high", "category": "billing", "sla_hours": 4}
    if "login" in text or "password" in text:
        return {"priority": "medium", "category": "account", "sla_hours": 8}
    if "delivery" in text or "late" in text:
        return {"priority": "medium", "category": "delivery", "sla_hours": 12}
    return {"priority": "low", "category": "general", "sla_hours": 24}


def suggest_support_action(category: str) -> str:
    """Suggest a support action based on ticket category."""
    actions = {
        "billing": "Verify transaction ID and check refund/payment status.",
        "account": "Ask user to reset password and verify registered phone/email.",
        "delivery": "Check courier tracking and promised delivery date.",
        "general": "Ask for more details and assign to support queue.",
    }
    return actions.get(category.lower(), actions["general"])


print(classify_ticket("My payment was deducted but order was not confirmed"))
print(suggest_support_action("billing"))

{'priority': 'high', 'category': 'billing', 'sla_hours': 4}
Verify transaction ID and check refund/payment status.


In [33]:
# LLM cell: requires OPENAI_API_KEY.

support_agent = AssistantAgent(
    "support_agent",
    model_client=create_model_client(),
    tools=[classify_ticket, suggest_support_action],
    system_message=(
        "You are a support triage assistant. Use tools to classify tickets and suggest actions. "
        "Return priority, category, SLA, and next action."
    ),
    reflect_on_tool_use=True,
)

await Console(
    support_agent.run_stream(
        task="Ticket: Customer says payment was deducted but the order confirmation did not come."
    )
)

---------- TextMessage (user) ----------
Ticket: Customer says payment was deducted but the order confirmation did not come.
---------- ToolCallRequestEvent (support_agent) ----------
[FunctionCall(id='call_1pPYwefwBiFQ85Nh5XQkh5wK', arguments='{"message": "Customer says payment was deducted but the order confirmation did not come."}', name='classify_ticket'), FunctionCall(id='call_rp8v5HH3xu241sF2auujWm3j', arguments='{"category": "Payment Issue"}', name='suggest_support_action')]
---------- ToolCallExecutionEvent (support_agent) ----------
[FunctionExecutionResult(content="{'priority': 'high', 'category': 'billing', 'sla_hours': 4}", name='classify_ticket', call_id='call_1pPYwefwBiFQ85Nh5XQkh5wK', is_error=False), FunctionExecutionResult(content='Ask for more details and assign to support queue.', name='suggest_support_action', call_id='call_rp8v5HH3xu241sF2auujWm3j', is_error=False)]
---------- TextMessage (support_agent) ----------
- **Priority:** High
- **Category:** Billing
- **S

TaskResult(messages=[TextMessage(id='a97d4f85-c931-439f-bb87-1ecbfbcc25da', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 19, 45, 356756, tzinfo=datetime.timezone.utc), content='Ticket: Customer says payment was deducted but the order confirmation did not come.', type='TextMessage'), ToolCallRequestEvent(id='07a7b078-94eb-4862-9098-f24fa49ee148', source='support_agent', models_usage=RequestUsage(prompt_tokens=128, completion_tokens=60), metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 19, 47, 640373, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_1pPYwefwBiFQ85Nh5XQkh5wK', arguments='{"message": "Customer says payment was deducted but the order confirmation did not come."}', name='classify_ticket'), FunctionCall(id='call_rp8v5HH3xu241sF2auujWm3j', arguments='{"category": "Payment Issue"}', name='suggest_support_action')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='14e28f56-2f64-4506-9e72-7c8ff06fdd71'

## 10. Mini Project 2: CSV Data Analyst Agent

This project simulates a common enterprise use case:

> A user uploads data and asks an agent to calculate insights.

The agent should use code execution instead of mental math.

In [34]:
# Create a sample CSV file for students.

import pandas as pd

csv_path = WORK_DIR / "student_sales_data.csv"

df = pd.DataFrame(
    [
        {"date": "2026-06-01", "channel": "walk-in", "orders": 14, "revenue": 21000},
        {"date": "2026-06-01", "channel": "whatsapp", "orders": 5, "revenue": 8500},
        {"date": "2026-06-02", "channel": "walk-in", "orders": 9, "revenue": 13500},
        {"date": "2026-06-02", "channel": "instagram", "orders": 6, "revenue": 12000},
        {"date": "2026-06-03", "channel": "walk-in", "orders": 11, "revenue": 17600},
        {"date": "2026-06-03", "channel": "whatsapp", "orders": 8, "revenue": 14400},
    ]
)

df.to_csv(csv_path, index=False)
print(csv_path.resolve())
df

C:\Users\LotusBlue\Coding\GenAI\Course\autogen_course_workspace\student_sales_data.csv


,date,channel,orders,revenue
0,2026-06-01,walk-in,14,21000
1,2026-06-01,whatsapp,5,8500
2,2026-06-02,walk-in,9,13500
3,2026-06-02,instagram,6,12000
4,2026-06-03,walk-in,11,17600
5,2026-06-03,whatsapp,8,14400


In [35]:
# LLM cell: requires OPENAI_API_KEY.

csv_agent = AssistantAgent(
    "csv_analysis_agent",
    model_client=create_model_client(),
    tools=[python_code_tool],
    system_message=(
        "You analyze CSV files using Python code execution. "
        "Always load the CSV with pandas, calculate from data, and explain in simple terms."
    ),
    reflect_on_tool_use=True,
)

await Console(
    csv_agent.run_stream(
        task=f'''
Read this CSV file: {csv_path.resolve()}
Calculate total revenue by channel, best channel, total orders,
and create a simple bar chart saved as channel_revenue.png.
'''
    )
)

---------- TextMessage (user) ----------

Read this CSV file: C:\Users\LotusBlue\Coding\GenAI\Course\autogen_course_workspace\student_sales_data.csv
Calculate total revenue by channel, best channel, total orders,
and create a simple bar chart saved as channel_revenue.png.

---------- ToolCallRequestEvent (csv_analysis_agent) ----------
[FunctionCall(id='call_h5VYQLbAZB8OVXfTQl4bIxiX', arguments='{"code":"import pandas as pd\\nimport matplotlib.pyplot as plt\\n\\n# Load the CSV file\\nfile_path = r\'C:\\\\Users\\\\LotusBlue\\\\Coding\\\\GenAI\\\\Course\\\\autogen_course_workspace\\\\student_sales_data.csv\'\\ndata = pd.read_csv(file_path)\\n\\n# Display the first few rows of the dataset to understand its structure\\ndata.head()"}', name='CodeExecutor')]
---------- ToolCallExecutionEvent (csv_analysis_agent) ----------
[FunctionExecutionResult(content='', name='CodeExecutor', call_id='call_h5VYQLbAZB8OVXfTQl4bIxiX', is_error=False)]
---------- TextMessage (csv_analysis_agent) ----------


TaskResult(messages=[TextMessage(id='0148a7f7-8791-4805-a6b6-a3d1bfbbd0c6', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 19, 48, 795669, tzinfo=datetime.timezone.utc), content='\nRead this CSV file: C:\\Users\\LotusBlue\\Coding\\GenAI\\Course\\autogen_course_workspace\\student_sales_data.csv\nCalculate total revenue by channel, best channel, total orders,\nand create a simple bar chart saved as channel_revenue.png.\n', type='TextMessage'), ToolCallRequestEvent(id='6b784fc7-b673-473b-9a70-4ae47799fd98', source='csv_analysis_agent', models_usage=RequestUsage(prompt_tokens=141, completion_tokens=87), metadata={}, created_at=datetime.datetime(2026, 6, 17, 19, 19, 50, 461524, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_h5VYQLbAZB8OVXfTQl4bIxiX', arguments='{"code":"import pandas as pd\\nimport matplotlib.pyplot as plt\\n\\n# Load the CSV file\\nfile_path = r\'C:\\\\Users\\\\LotusBlue\\\\Coding\\\\GenAI\\\\Course\\\\autogen_co

## 11. Common Errors and Fixes

### Error: `OPENAI_API_KEY is not set`

Set the key before running LLM cells:

```python
import os
os.environ["OPENAI_API_KEY"] = "your-key"
```

For real use, prefer setting it outside the notebook.

### Error: package import failed

Re-run the `%pip install` cell and restart the kernel.

### Error: Docker not running

Skip the Docker section or start Docker Desktop.

### Agent does not call tool

Improve the system message and tool docstring. Make tool names and descriptions specific.

### Code execution is risky

Use Docker, limit permissions, review generated code, and never expose secrets in the execution environment.

## 12. Student Exercises

### Exercise A: Add a new retail tool

Create a tool called `recommend_size(height_cm, fit_preference)` that recommends S/M/L/XL.

### Exercise B: Add validation

Modify `calculate_bill` so quantity must be between 1 and 20.

### Exercise C: Add a real-world API

Create a weather or currency conversion tool using an API. Keep the API key outside code.

### Exercise D: Safer code execution

Modify the safety checker to block file deletion and network calls.

### Exercise E: Build a business agent

Create an AutoGen assistant for one domain:

- Boutique assistant
- Clinic reception assistant
- Restaurant booking assistant
- School enquiry assistant
- Warehouse stock assistant

## 13. Capstone Assignment

Build a **Retail Operations Agent** using AutoGen.

### Requirements

1. It should answer inventory questions using tools.
2. It should calculate offers and bills using tools.
3. It should analyze a CSV using code execution.
4. It should produce a final recommendation.
5. It should include at least one safety check before code execution.

### Expected output

- A working notebook
- A short explanation of each tool
- Screenshot/output of at least three successful runs
- One paragraph explaining risks and safety controls

## 14. Reference Links

- AutoGen GitHub: https://github.com/microsoft/autogen
- AutoGen Tools documentation: https://microsoft.github.io/autogen/stable/user-guide/core-user-guide/components/tools.html
- AutoGen Python code execution tool: https://microsoft.github.io/autogen/stable/reference/python/autogen_ext.tools.code_execution.html
- AutoGen command line code executors: https://microsoft.github.io/autogen/stable/user-guide/core-user-guide/components/command-line-code-executors.html
- AutoGen AgentChat agents: https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/agents.html
- AutoGen SelectorGroupChat: https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/selector-group-chat.html